# 预算：为会话的支出设置上限

无人看管的智能体在成本上没有天然的停止点。一个带网页搜索的调研任务可能不知不觉多出一百次抓取。一个编码循环可能反复重试某个不稳定的测试直到天亮。没有上限，你只能在收到账单时才发现。

会话预算是对单个会话强制执行的支出上限。你在创建会话时设置一个最高的列表价成本。平台按公开列表价跟踪该会话中每个线程的词元成本，当累计总额达到上限时，会话会以停止原因 `budget_reached` 进入空闲状态。它的文件、工具状态和对话都保持完整。提高上限，工作就会从中断处继续。

本笔记本将讲解：

- 在 `sessions.create` 上设置 `budget`
- 读取 `usage.list_cost`，以及流在回合结束时送达的 `session.usage` 快照
- `budget_reached` 停止，以及暂停的会话会保留什么
- 用 `sessions.update` 提高、降低和移除上限

## 1. 设置客户端

预算依赖标准的 `managed-agents-2026-04-01` 测试版请求头。`budget` 参数以及 `session.usage` 上的成本字段需要 `anthropic>=0.121.0`。

In [1]:
%%capture
%pip install -qU "anthropic>=0.121.0" python-dotenv

In [2]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-5")
client = anthropic.Anthropic()

## 2. 构建一个有超支空间的智能体

本示例中的智能体负责撰写一份竞争格局简报。它拥有网页搜索和网页抓取能力，而它的提示词中没有任何内容限制它阅读多少来源。正是这种开放性构成了预算的用武之地。

In [3]:
env = client.beta.environments.create(
    name="budget-demo",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)

analyst = client.beta.agents.create(
    name="market_analyst",
    description="Writes sourced competitive landscape briefs.",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-cap-session-spend"},
    model={"id": MODEL},
    system="""You write competitive landscape briefs for product teams.
Given a market, use web search and web fetch to find the notable vendors, their
positioning, and recent moves. Read primary sources rather than aggregators.
Write the brief to /mnt/session/outputs/brief.md with a Sources section listing every
URL you relied on. Keep researching until you are confident the brief is complete.""",
    tools=[
        {
            "type": "agent_toolset_20260401",
            "configs": [{"name": "web_search"}, {"name": "web_fetch"}],
        }
    ],
    betas=BETAS,
)
print(f"{analyst.name}: {analyst.id} v{analyst.version}")

market_analyst: agent_staging_014U3BWzuiSNDwfzubN38hnz v1


## 3. 带预算创建会话

`sessions.create` 上的 `budget` 字段用于设置上限：

```json
{"type": "limit", "max_list_cost": {"currency": "USD", "amount": "10"}}
```

在你选定数值之前，有几点值得了解：

- `amount` 是以该货币最小单位表示的整数字符串：`"50"` 是五十美分，`"2500"` 是 $25.00。它始终是字符串，且拒绝小数美分，因此永远不会应用浮点舍入。`USD` 是唯一接受的货币，API 返回的每一个成本金额（包括 `usage.list_cost`）都使用相同的编码。
- 上限按公开列表价计算模型词元成本，在会话中的每个线程之间累加，包括子智能体线程。无论是否有任何协商折扣，都按列表价计算，因此上限会在你的实际账单之前或恰好此时触发，而且你可以用 `session.usage` 加上公开价目表复现这个数字。
- 会话能够运行的每个模型都必须有公开列表价。如果某个模型没有，创建会以 `model_not_budgetable` 失败。
- 省略 `budget`，会话就没有上限。预算只能在创建时附加：没有预算创建的会话以后永远无法获得预算，所以要事先决定。

这里的上限是十美分，故意设得很低，以便停止在几分钟内出现。下面的 `usd` 辅助函数会把最小单位金额渲染为美元，供后续所有打印使用。

In [4]:
from decimal import Decimal


def usd(money) -> str:
    """Render an integer minor-unit amount ("50" = fifty cents) as dollars."""
    return f"${Decimal(money.amount) / 100:.2f}"


session = client.beta.sessions.create(
    agent=analyst.id,
    environment_id=env.id,
    title="Landscape brief: observability platforms",
    budget={
        "type": "limit",
        "max_list_cost": {"currency": "USD", "amount": "10"},
    },
    initial_events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": "Write a competitive landscape brief on the observability "
                    "platform market: the main vendors, how each positions itself, "
                    "and any notable moves in the last year.",
                }
            ],
        }
    ],
    betas=BETAS,
)
print(session.id, session.status)
print("budget:", usd(session.budget.max_list_cost))

sesn_staging_01ELJgA2vqTR6EmjX8viwHZi running
budget: $0.10


## 4. 观察支出累积

除了常见的 `agent.message` 和 `agent.tool_use` 事件，每当会话进入 `idle` 状态，流都会送达一个 `session.usage` 快照。它包含累计的词元计数、被跟踪的 `list_cost` 以及配置的 `budget`，因此回合一结束，无需再次调用 `sessions.retrieve`，你就知道它花了多少。

循环在 `session.status_idle` 时退出。存在预算时，有两种停止原因会结束回合：智能体完成时的 `end_turn`，以及上限先触发的 `budget_reached`。

In [5]:
stop_reason = None

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "agent.tool_use":
            print(f"[{ev.name}] {str(ev.input)[:80]}")
        elif ev.type == "session.usage":
            print(f"  usage: {usd(ev.usage.list_cost)} of {usd(ev.budget.max_list_cost)}")
        elif ev.type == "session.status_idle":
            stop_reason = ev.stop_reason.type
            print(f"[idle] stop_reason={stop_reason}")
            break

assert stop_reason == "budget_reached", (
    f"expected budget_reached, got {stop_reason}: lower the cap in step 3 "
    "or give the agent a bigger task"
)

[repl] {'script': '\nlet queries = [\n  "Datadog positioning observability platform 202
[web_search] {'query': 'Datadog positioning observability platform 2025'}
[web_search] {'query': 'New Relic observability platform positioning 2025'}
[web_search] {'query': 'Dynatrace observability platform AI 2025'}
[web_search] {'query': 'Splunk Cisco observability 2025'}
[web_search] {'query': 'Grafana Labs observability platform 2025 announcement'}
[web_search] {'query': 'Elastic observability 2025'}
[web_search] {'query': 'Honeycomb observability 2025'}
[web_search] {'query': 'Chronosphere observability 2025 acquisition'}
[web_search] {'query': 'Datadog acquisition 2024 2025'}
[web_search] {'query': 'Grafana Labs acquisition 2024 2025'}


  usage: $0.12 of $0.10
[idle] stop_reason=budget_reached


## 5. 检查暂停的会话

`budget_reached` 是暂停，不是失败。会话处于 `idle` 状态，它的 `usage` 反映了到目前为止的全部支出，容器中仍然保存着智能体在上限触发前写入的任何内容。什么都不需要重跑。

上限是在模型请求之间强制执行的，因此记录的 `list_cost` 可能略微超过 `max_list_cost`：越过界线的那次请求会完成，然后会话才暂停。设定上限时要把这一次请求的余量考虑在内。

In [6]:
paused = client.beta.sessions.retrieve(session.id, betas=BETAS)

print("status:      ", paused.status)
print("list cost:   ", usd(paused.usage.list_cost))
print("cap:         ", usd(paused.budget.max_list_cost))
print("tokens:       in", paused.usage.input_tokens, "out", paused.usage.output_tokens)
print("active secs: ", round(paused.usage.active_seconds, 1))

status:       idle
list cost:    $0.12
cap:          $0.10
tokens:       in 471 out 606
active secs:  13.7


事件日志显示了智能体进展到哪一步：它做出的工具使用，以及如果它已开始叙述，它的最后一条消息。一个深入调研中的智能体在上限触发时可能还没有写下任何内容，所以要把消息视为可选项。

In [7]:
events = list(client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS))
tool_calls = [ev for ev in events if ev.type == "agent.tool_use"]
messages = [ev for ev in events if ev.type == "agent.message"]

print(f"tool calls before the cap: {len(tool_calls)}")
if messages:
    last = messages[-1]
    print("".join(b.text for b in last.content if b.type == "text")[:600])
else:
    print("(no agent.message yet: the agent was still gathering sources when the cap fired)")

tool calls before the cap: 11
(no agent.message yet: the agent was still gathering sources when the cap fired)


## 6. 提高上限并让它完成

`sessions.update` 接受相同结构的 `budget`。把 `max_list_cost` 提高到已消耗成本之上即可解除暂停，会话会自行从停止时的状态恢复被中断的回合。

把上限降低到等于或低于会话已花费的金额会被拒绝，返回 400：`max_list_cost` 必须保持在已消耗列表价成本之上，因此更新永远不会让会话与自己的历史相互卡死。第一次调用演示了这种拒绝，第二次才真正提高上限。提高之后无需发送后续消息；会话回到 `running` 状态并接续被中断的回合。

In [8]:
try:
    client.beta.sessions.update(
        session.id,
        budget={"type": "limit", "max_list_cost": {"currency": "USD", "amount": "1"}},
        betas=BETAS,
    )
except anthropic.BadRequestError as e:
    print("lower rejected:", e.message)

client.beta.sessions.update(
    session.id,
    budget={"type": "limit", "max_list_cost": {"currency": "USD", "amount": "500"}},
    betas=BETAS,
)
print("cap raised to $5.00")

lower rejected: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': "`budget.max_list_cost` must be greater than the session's consumed list cost"}, 'request_id': 'req_staging_011CdiPi8iPJKWdiuurZuDkB'}


cap raised to $5.00


In [9]:
with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "session.status_running":
            print("[resumed]")
        elif ev.type == "agent.tool_use":
            print(f"[{ev.name}]")
        elif ev.type == "session.status_idle":
            print(f"[idle] stop_reason={ev.stop_reason.type}")
            break

done = client.beta.sessions.retrieve(session.id, betas=BETAS)
print("final list cost:", usd(done.usage.list_cost), "of cap", usd(done.budget.max_list_cost))

[resumed]


[repl]
[web_search]
[web_search]
[web_search]
[web_search]
[web_search]
[web_search]
[web_search]
[web_search]
[web_search]
[web_search]


[repl]
[web_fetch]
[web_fetch]
[web_fetch]
[web_fetch]
[web_fetch]
[web_fetch]
[web_fetch]
[web_fetch]


[repl]
[web_fetch]


[web_fetch]
[web_fetch]


...

[write]


[write]


[bash]


[idle] stop_reason=end_turn


final list cost: $1.83 of cap $5.00


## 7. 阅读简报

智能体在上下文完整的情况下重新拾起任务并完成了写作。从事件日志中取出它生成的文件。

In [10]:
brief = ""
for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if (
        ev.type == "agent.tool_use"
        and ev.name == "write"
        and ev.input.get("file_path", "").endswith("brief.md")
    ):
        brief = ev.input["content"]  # keep the last write, in case of revisions

print(brief[:1500] + ("\n..." if len(brief) > 1500 else ""))

# Competitive Landscape Brief: Observability Platform Market

*Scope: commercial platforms for metrics, logs, traces, and related telemetry (APM, infrastructure monitoring, log management, digital experience monitoring) used by engineering, IT operations, and SRE teams. Window for "recent moves": roughly the last twelve months of vendor activity found in primary sources (mid‑2025 through mid‑2026).*

## 1. Market shape

Observability has consolidated into a market with a handful of large, broad "full platform" vendors (Datadog, Splunk/Cisco, Dynatrace, New Relic, Elastic, Grafana Labs), a tier of cost‑ and scale‑focused challengers (Chronosphere, Coralogix, Honeycomb, SolarWinds, LogicMonitor, Sumo Logic), hyperscaler-native tooling bundled into the cloud platforms (AWS CloudWatch, Azure Monitor, Google Cloud Observability), and an open standard — OpenTelemetry — that now sits underneath almost every vendor's ingestion layer and was elevated to CNCF "graduated" status, cementing it as 

## 8. 移除预算

在更新时传入 `budget=None` 会清除上限，会话从那一刻起不再有上限。完全省略 `budget` 则保留当前上限。这是两种不同的请求：`None` 是显式移除，缺省是保留。

移除是永久性的，原因与预算只能在创建时附加相同：一旦会话没有预算，它就无法再获得预算。你可以随意提高和降低上限，但要把移除视为一扇单行道。

In [11]:
uncapped = client.beta.sessions.update(session.id, budget=None, betas=BETAS)
print("budget:", uncapped.budget)

budget: None


## 9. 清理

In [12]:
from utilities import wait_for_idle_status

wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id, betas=BETAS)
client.beta.environments.archive(env.id, betas=BETAS)
print("archived")

archived


## 预算适合用在何处

预算是会话级的控制手段。它约束的是一次运行，而不是一个组织、一个工作区或一天的流量，因此它与你已在这些层级设置的支出限制叠加生效，而不是取代它们。每当会话在无人监督的情况下运行，就应考虑使用它：cron 驱动的部署、webhook 触发的智能体，以及协调器的扇出取决于数据的长时间多智能体作业。

对于你不进行流式连接的会话群组，改为订阅 `session.budget_reached` webhook，而不必保持连接打开。它在会话达到上限时触发一次，载荷中会指明会话，以便监督进程决定是提高上限还是让会话保持暂停。webhook 模式在 [﷐1﷑](CMA_operate_in_production.ipynb) 中有讲解。